# DataFrame ビルダ: クエリを Python の値として扱う

`db.table(...)` はメソッド呼び出しで組み立てる**遅延**クエリの入口です。SQL 文字列を書く
代わりに動詞をつないでいき、`.collect()` のような終端の呼び出しまで何も走りません。これは
第2のエンジンではなくコンパイラです。どの動詞も `db.sql()` を通る SQL に落ちるので、
組み立てたクエリが見るセッションもテーブル関数もバージョンの固定も、自分で書いたであろう
文字列とまったく同じです。そして `.sql()` を呼べば、何が生成されたかがそのまま見えます。

リサーチの現場での見返りは、生成するクエリにあります。窓幅や列をループで掃引する
ファクターライブラリは、いまなら f 文字列で SQL を組み立てているはずです。クォートのバグ
や `'` の混入が住み着くのはそこです。ビルダなら識別子のクォートは1か所に集約され、
組み立て途中のパイプラインは、持ち回して伸ばして再利用できるただの Python の値になります。

In [1]:
import h5i_db
from h5i_db import col, count_star, lit, sql_expr, time_bucket, vwap, when

import cookbook_utils as cu

db = h5i_db.Database(cu.fresh_db("00_dataframe_builder"), create=True)

trades = cu.make_trades(symbols=["AAPL", "MSFT", "NVDA"], days=3, trades_per_day=20_000)
db.create_table("trades", trades.schema, time_column="ts", sort_key=["ts", "symbol"])
db.append("trades", trades)

prices = cu.make_daily_prices(days=500)  # 50 names x 500 sessions
db.create_table("prices", prices.schema, time_column="ts", sort_key=["ts", "symbol"])
db.append("prices", prices)

print(f"trades: {len(trades):,} rows   prices: {len(prices):,} rows")

trades: 195,277 rows   prices: 25,000 rows


## 1. フレームとは、まだ走っていないクエリ

`db.table("trades")` はテーブル全体を出発点にします。動詞は**新しい**フレームを返すので、
元は書き換わらず、途中まで組んだパイプラインをそのまま再利用できます。`.sql()` が文字列に
し、`.collect()` が実行します。

In [2]:
liquid = db.table("trades").filter(col("symbol").is_in(["AAPL", "NVDA"]))
print(liquid.sql())

SELECT *
FROM "trades"
WHERE "symbol" IN ('AAPL', 'NVDA')


定番の OHLCV ロールアップを組み立てるとこうなります。`group_by(...).agg(...)` はキーを
集約と並べて射影します。`.first("ts")` と `.last("ts")` は `first_value(x ORDER BY ts)`
の書き方そのもので、自己結合なしにバーの始値と終値を返します。

In [3]:
bars = (
    db.table("trades")
    .group_by(time_bucket("1m", col("ts")).alias("bar"), "symbol")
    .agg(
        col("price").first("ts").alias("open"),
        col("price").max().alias("high"),
        col("price").min().alias("low"),
        col("price").last("ts").alias("close"),
        col("size").sum().alias("volume"),
        vwap(col("price"), col("size")).alias("vwap"),
    )
    .sort(["bar", "symbol"])
)
print(bars.sql())

SELECT time_bucket('1m', "ts") AS "bar", "symbol", first_value("price" ORDER BY "ts") AS "open", max("price") AS "high", min("price") AS "low", last_value("price" ORDER BY "ts") AS "close", sum("size") AS "volume", vwap("price", "size") AS "vwap"
FROM "trades"
GROUP BY "bar", "symbol"
ORDER BY "bar", "symbol"


In [4]:
bars.to_pandas().head(6)

,bar,symbol,open,high,low,close,volume,vwap
0,2026-06-01 13:30:00+00:00,AAPL,265.05,265.23,265.01,265.17,10308,265.137426
1,2026-06-01 13:30:00+00:00,MSFT,363.06,363.17,362.76,363.12,9544,362.944815
2,2026-06-01 13:30:00+00:00,NVDA,319.22,319.43,318.97,319.29,12402,319.220094
3,2026-06-01 13:31:00+00:00,AAPL,265.16,265.28,265.13,265.22,11005,265.199838
4,2026-06-01 13:31:00+00:00,MSFT,363.11,363.11,362.73,362.75,10846,362.922534
5,2026-06-01 13:31:00+00:00,NVDA,319.38,319.38,318.78,319.31,10606,319.094032


## 2. 式

`col(name)` が列、`lit(value)` が定数で、四則演算と比較はそこから積み上がります。早めに
出会っておきたい罠が2つあります。

- Python では `and` / `or` / `not` を多重定義できないので、論理は `&`、`|`、`~` を使います。
  これらは比較より*強く*結合するため、比較のたびに括弧が要ります。
- 式が保つのは Python ではなく **SQL** の意味です。整数列どうしの `/` は整数除算になります。
  本当の除算がほしいならキャストしてください。

In [5]:
signed = (
    db.table("trades")
    .filter((col("price") > 0) & (col("size") >= 100))
    .select(
        "ts",
        "symbol",
        "price",
        notional=col("price") * col("size"),
        lots=col("size").cast("DOUBLE") / 100,
        direction=when(col("side") == "B").then(lit(1)).otherwise(lit(-1)),
    )
)
print(signed.sql())

SELECT "ts", "symbol", "price", "price" * "size" AS "notional", CAST("size" AS DOUBLE) / 100 AS "lots", CASE WHEN "side" = 'B' THEN 1 ELSE -1 END AS "direction"
FROM "trades"
WHERE "price" > 0 AND "size" >= 100


In [6]:
signed.to_pandas().head(4)

,ts,symbol,price,notional,lots,direction
0,2026-06-02 13:36:14.322833+00:00,MSFT,370.91,37091.0,1.0,-1
1,2026-06-02 13:36:14.413593+00:00,AAPL,266.56,79968.0,3.0,-1
2,2026-06-02 13:36:14.561120+00:00,NVDA,315.54,31554.0,1.0,1
3,2026-06-02 13:36:14.848470+00:00,MSFT,371.01,37101.0,1.0,1


識別子は常にクォートされるので大文字小文字が保たれ（素の SQL なら小文字に畳まれる
`Symbol` という名前のフィールドも `col("Symbol")` で見つかります）、文字列リテラルは
あくまで文字列で、構文になることはありません。

In [7]:
print(db.table("trades").filter(col("symbol") == "'; DROP TABLE trades; --").sql())

SELECT *
FROM "trades"
WHERE "symbol" = '''; DROP TABLE trades; --'


## 3. パイプラインが SQL になるまで

たいていのパイプラインは平らな `SELECT` 1つに落ちます。独立した `with_columns` は1つに
まとまり、*元からある*列での絞り込みは同じ `WHERE` に残ります。一方、前の段が**計算した**
列を読む段は自分の階層を持ちます。SQL は `WHERE` を選択リストの兄弟ではなく `FROM` に
対して解決するからです。集約と `LIMIT` と `DISTINCT` も階層を閉じます。後続はその出力に
対して働くためです。

In [8]:
movers = (
    db.table("prices")
    .with_columns(ret=col("close") / col("open") - 1)
    .filter(col("ret") > 0.01)  # reads a computed column -> subquery
    .sort("ret", descending=True)
    .limit(5)
)
print(movers.sql())

SELECT *
FROM (
  SELECT *, "close" / "open" - 1 AS "ret"
  FROM "prices"
) AS "_s1"
WHERE "ret" > 0.01
ORDER BY "ret" DESC
LIMIT 5


In [9]:
movers.to_pandas()

,ts,symbol,open,high,low,close,volume,ret
0,2023-07-06 20:00:00+00:00,STK003,192.62,195.55,192.03,195.05,514942,0.012616
1,2024-08-14 20:00:00+00:00,STK020,131.80,133.81,131.03,133.38,270545,0.011988
2,2023-01-09 20:00:00+00:00,STK016,30.51,30.94,30.42,30.85,480125,0.011144
3,2023-01-06 20:00:00+00:00,STK008,96.41,97.76,96.36,97.47,339522,0.010995
4,2023-07-04 20:00:00+00:00,STK043,342.13,347.28,341.83,345.81,292791,0.010756


身につける価値があるのは、階層が*どこで*閉じるかの感覚です。次の段に何が見えるかがそれで
決まります。パイプラインが平らなあいだは、動詞は元のテーブルまで届きます。
`select("ts", "symbol").sort("close")` がきちんと解決するのは、SQL の `ORDER BY` が
見にいく先も同じく `FROM` だからです。集約が階層を閉じたあとは、その列は本当に消えていて、
エンジンがそう言ってきます。

In [10]:
try:
    db.table("prices").group_by("symbol").agg(count_star().alias("n")).sort("close").collect()
except h5i_db.H5iError as e:
    print(f"{type(e).__name__}: {str(e)[:180]}")

H5iError: [query] Error during planning: Column in ORDER BY must be in GROUP BY or an aggregate function: While expanding wildcard, column "prices.close" must appear in the GROUP BY clause o


## 4. 見返り: 生成するクエリ

ビルダが本領を発揮するのはここです。複数の遡及期間の掃引は、フレームを回すだけの Python の
ループになります。文字列を切り貼りする必要はなく、それぞれのフレームは持って名前を付けて
再利用できる値です。ここでは、価格とその移動平均との乖離を3つの窓で見ます。

rolling 系のメソッドは `window` と `order_by` を取り、任意で `partition_by` も取ります。
SQL の糖衣構文 `rolling_avg` と違って本物の `PARTITION BY` を伴うので、複数銘柄のテーブル
でも銘柄が**混ざりません**。

In [11]:
WINDOWS = (5, 20, 60)

base = db.table("prices").filter(col("symbol").is_in(["STK000", "STK001", "STK002"]))

ma_gap = base.with_columns(
    **{
        f"gap_{n}d": col("close") / col("close").rolling_mean(n, order_by="ts", partition_by="symbol") - 1
        for n in WINDOWS
    }
)
print(ma_gap.sql())

SELECT *, "close" / avg("close") OVER (PARTITION BY "symbol" ORDER BY "ts" ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) - 1 AS "gap_5d", "close" / avg("close") OVER (PARTITION BY "symbol" ORDER BY "ts" ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) - 1 AS "gap_20d", "close" / avg("close") OVER (PARTITION BY "symbol" ORDER BY "ts" ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) - 1 AS "gap_60d"
FROM "prices"
WHERE "symbol" IN ('STK000', 'STK001', 'STK002')


In [12]:
ma_gap.select("ts", "symbol", *[f"gap_{n}d" for n in WINDOWS]).sort(["ts", "symbol"]).to_pandas().tail(6)

,ts,symbol,gap_5d,gap_20d,gap_60d
1494,2024-11-28 20:00:00+00:00,STK000,0.000589,0.046048,0.064262
1495,2024-11-28 20:00:00+00:00,STK001,-0.002695,0.028779,0.034177
1496,2024-11-28 20:00:00+00:00,STK002,-0.019274,-0.018988,0.043117
1497,2024-11-29 20:00:00+00:00,STK000,0.014864,0.060963,0.083427
1498,2024-11-29 20:00:00+00:00,STK001,-0.017498,0.010656,0.018326
1499,2024-11-29 20:00:00+00:00,STK002,0.015216,0.017048,0.080730


クロスセクションの演算子は、値を*同じ時点の*仲間と比べて順位づけるので、比較する範囲を
引数に取ります。z スコア化したいくつかのシグナルを1つの合成値にまとめる形は、そのまま
ファクター構築の骨格です。

In [13]:
combo = (
    db.table("prices")
    .with_columns(
        z_ret=(col("close") / col("open") - 1).cs_zscore(partition_by="ts"),
        z_vol=col("volume").cast("DOUBLE").cs_zscore(partition_by="ts"),
    )
    .with_columns(score=(col("z_ret") - col("z_vol")) / 2)
    .select("ts", "symbol", "z_ret", "z_vol", "score")
    .sort(["ts", "score"], descending=[False, True])
)
combo.to_pandas().head(5)

,ts,symbol,z_ret,z_vol,score
0,2023-01-02 20:00:00+00:00,STK034,1.935241,-0.691824,1.313533
1,2023-01-02 20:00:00+00:00,STK040,0.791453,-1.559526,1.175490
2,2023-01-02 20:00:00+00:00,STK021,0.728941,-1.457394,1.093167
3,2023-01-02 20:00:00+00:00,STK001,1.070606,-1.111451,1.091028
4,2023-01-02 20:00:00+00:00,STK005,0.683265,-1.265949,0.974607


## 5. バージョンの固定とジョイン

読み取り点は `db.table()` にそのまま渡り、`h5i()` に落ちます。固定したビルダのクエリは、
手で書いた SQL とまったく同じくソースの時点で束縛されます。`.join()` は両側を `l` と `r`
という別名のサブクエリとして描き、この別名が特定の側に触れるための約束事になります。
おかげで「同じクエリを N 個のバージョンに当てる」比較が、関数呼び出し1つで済みます。

In [14]:
db.append("trades", cu.make_trades(symbols=["AAPL", "MSFT", "NVDA"], days=1, start="2026-06-04", seed=8))


def per_symbol(version=None):
    return db.table("trades", version=version).group_by("symbol").agg(
        count_star().alias("n"), col("ts").max().alias("last_ts")
    )


drift = per_symbol(1).join(per_symbol(), on="symbol").select(
    symbol=col("symbol", relation="l"),
    trades_added=col("n", relation="r") - col("n", relation="l"),
)
print(drift.sql())

SELECT "l"."symbol" AS "symbol", "r"."n" - "l"."n" AS "trades_added"
FROM (
  SELECT "symbol", count(*) AS "n", max("ts") AS "last_ts"
  FROM h5i('trades', 1)
  GROUP BY "symbol"
) AS "l"
INNER JOIN (
  SELECT "symbol", count(*) AS "n", max("ts") AS "last_ts"
  FROM "trades"
  GROUP BY "symbol"
) AS "r"
  ON "l"."symbol" = "r"."symbol"


In [15]:
drift.sort("symbol").to_pandas()

,symbol,trades_added
0,AAPL,24438
1,MSFT,14065
2,NVDA,22443


`.join_asof()` は `asof_join` テーブル関数に落ちます。この関数が取るのは*テーブル名*で、
どちらも最新版として読むので、すでに動詞を当てた側や固定した側をビルダは黙って無視せず
拒否します。絞り込みはジョインのあとに置いてください。

In [16]:
tape, quotes = cu.make_trades_and_quotes(days=2)  # shared base prices
for name, data in (("tape", tape), ("quotes", quotes)):
    db.create_table(name, data.schema, time_column="ts", sort_key=["ts", "symbol"])
    db.append(name, data)

try:
    db.table("tape").filter(col("symbol") == "AAPL").join_asof(db.table("quotes"), on="ts", by="symbol")
except h5i_db.InvalidInputError as e:
    print(f"{type(e).__name__}: {e}\nhint: {e.hint}")

InvalidInputError: [invalid_input] join_asof() needs a plain table on the left side, but operations have already been applied
hint: join first and filter afterwards, or materialise the side with .collect() and write it back


In [17]:
tq = (
    db.table("tape")
    .join_asof(db.table("quotes"), on="ts", by="symbol", tolerance=5_000_000)
    .filter(col("symbol") == "AAPL")
    .select("ts", "symbol", "price", "bid", "ask", mid=(col("bid") + col("ask")) / 2)
)
print(tq.sql())

SELECT "ts", "symbol", "price", "bid", "ask", ("bid" + "ask") / 2 AS "mid"
FROM asof_join('tape', 'quotes', 'ts', 'ts', 'symbol', 'backward', 5000000)
WHERE "symbol" = 'AAPL'


In [18]:
tq.to_pandas().head(4)

,ts,symbol,price,bid,ask,mid
0,2026-06-01 17:02:22.097763+00:00,AAPL,264.84,266.15,266.21,266.18
1,2026-06-01 17:02:22.474044+00:00,AAPL,264.88,266.19,266.21,266.20
2,2026-06-01 17:02:22.489221+00:00,AAPL,264.88,266.19,266.21,266.20
3,2026-06-01 17:02:23.621502+00:00,AAPL,264.91,266.17,266.19,266.18


## 6. 非常口と、SQL へ戻る扉

動詞だけで SQL の全機能を覆うことは、意図して目標にしていません。`sql_expr()` は式を
受け付ける場所ならどこにでも生の断片を落とせます。テキストはそのまま挿入されるので、
クォートの責任が自分にある唯一の場所でもあります。

In [19]:
tails = (
    db.table("prices")
    .group_by("symbol")
    .agg(
        p01=sql_expr("approx_percentile_cont(close, 0.01)"),
        p99=sql_expr("approx_percentile_cont(close, 0.99)"),
    )
    .sort("symbol")
    .limit(4)
)
tails.to_pandas()

,symbol,p01,p99
0,STK000,25.7775,68.6225
1,STK001,172.5800,259.4775
2,STK002,209.8525,357.6050
3,STK003,110.8075,215.2300


非常口のうち、いちばん手を伸ばすことになるのは `lag` です。`.lag()` メソッドはありません
が、`sql_expr` の断片はウィンドウ化できるので、他の集約と同じように `.over()` を取ります。
これで `lag`、`lead`、`row_number` をはじめ SQL のウィンドウ関数一式が使えます。日次
リターンという、このクックブックで最も多く出てくる形を見てみます。

In [20]:
PREV_CLOSE = sql_expr("lag(close)").over(partition_by="symbol", order_by="ts")

rets = (
    db.table("prices")
    .with_columns(prev_close=PREV_CLOSE)
    .with_columns(ret=col("close") / col("prev_close") - 1)
    .filter(col("ret").is_not_null())
    .select("ts", "symbol", "ret")
)
print(rets.sql())

SELECT "ts", "symbol", "ret"
FROM (
  SELECT *, "close" / "prev_close" - 1 AS "ret"
  FROM (
    SELECT *, lag(close) OVER (PARTITION BY "symbol" ORDER BY "ts") AS "prev_close"
    FROM "prices"
  ) AS "_s1"
) AS "_s2"
WHERE "ret" IS NOT NULL


In [21]:
rets.sort(["ts", "symbol"]).to_pandas().head(4)

,ts,symbol,ret
0,2023-01-03 20:00:00+00:00,STK000,0.015468
1,2023-01-03 20:00:00+00:00,STK001,-0.001578
2,2023-01-03 20:00:00+00:00,STK002,0.025035
3,2023-01-03 20:00:00+00:00,STK003,0.015922


`with_columns` が2段になっている点に注目してください。`ret` は前の段が計算した
`prev_close` を読むので、ビルダは解決できない SQL を吐く代わりに階層を閉じます。断片を
`PREV_CLOSE` のような Python の名前に1度だけ束ねて使い回す習慣が、ファクターライブラリを
正直に保ってくれます。

パイプラインがビルダの手に余るようになったら、`.sql()` がクエリを渡してくれるので、それを
`db.sql()` に貼って続きを書けます。2つの面は、あいだに扉のある1つの仕組みです。生成される
SQL は決定的なので、スナップショットテストにかけても差分を取っても構いません。

対応する動詞がなく、文字列のほうが素直に読める場面は `db.sql()` に残ります。`UNION ALL`、
深い多段 CTE、スカラサブクエリ、そして `gapfill` / `resample` / `tail` の各テーブル関数です。
2つの読み取り点をラベル付きで1つの結果に積むのが、日常的な例です。

In [22]:
db.sql(
    """
    SELECT 'version 1' AS read_point, count(*) AS rows FROM h5i('trades', 1)
    UNION ALL
    SELECT 'latest',                  count(*)         FROM trades
    """
).to_pandas()

,read_point,rows
0,latest,256223
1,version 1,195277


## まとめ

- `db.table(...)` は遅延クエリです。動詞は新しいフレームを返し、`.collect()` や
  `.to_pandas()` まで何も走りません。`.sql()` でコンパイル結果の SQL が見えます。
- ビルダは `db.sql()` の上のコンパイラであって、第2のエンジンではありません。セッションも
  テーブル関数も `h5i()` によるバージョン固定も、すべて同じものです。
- 論理には `&`、`|`、`~` を使い、式が SQL の意味を持つことを忘れないでください。整数の
  `/` は切り捨てます。
- 手を伸ばすべきなのはクエリを**生成する**とき、つまり窓幅や列をループで掃引するような
  場面です。f 文字列の SQL がクォートのバグを招くところが、ちょうどビルダの持ち場になります。
  1度しか書かないクエリなら、素の SQL のほうが短いことも多いはずです。
- `rolling_*` と `cs_*` のメソッドは本物の `PARTITION BY` を伴います。SQL の糖衣構文
  `rolling_avg` のほうは全体をまたぐ行数ベースの窓で、そこが違います。
- `sql_expr()` が非常口、`.sql()` が戻り道です。どちらの面も二級市民ではありません。

In [23]:
db.close()